In [1]:
from utils.std_model import base_model
from langchain_core.runnables import RunnablePassthrough, RunnableBranch
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

chatLLM = base_model()


## --- 定义模拟子智能体处理程序（相当于 ADK 的 sub_agents）---
def booking_handler(request: str) -> str:
    """模拟预订智能体请求。"""
    print("\n--- 委托给预订处理程序 ---")
    return f"预订处理程序处理了请求：'{request}'。结果：模拟预订操作。"


def info_handler(request: str) -> str:
    """模拟信息智能体请求。"""
    print("\n--- 委托给信息处理程序 ---")
    return f"信息处理程序处理了请求：'{request}'。结果：模拟信息检索。"


def unclear_handler(request: str) -> str:
    """处理无法委托的请求。"""
    print("\n--- 处理不清楚的请求 ---")
    return f"协调器无法委托请求：'{request}'。请澄清。"


coordinator_router_prompt = ChatPromptTemplate.from_messages([
    ("system", """分析用户的请求并确定哪个专家处理程序应处理它。
     - 如果请求与预订航班或酒店相关，
        输出 'booker'。
     - 对于所有其他一般信息问题，输出 'info'。
     - 如果请求不清楚或不适合任一类别，
        输出 'unclear'。
     只输出一个词：'booker'、'info' 或 'unclear'。"""),
    ("user", "{request}")
])

coordinator_router_chain = coordinator_router_prompt | chatLLM | StrOutputParser()

branches = {
    "booker": RunnablePassthrough.assign(output=lambda x: booking_handler(x['request']['request'])),
    "info": RunnablePassthrough.assign(output=lambda x: info_handler(x['request']['request'])),
    "unclear": RunnablePassthrough.assign(output=lambda x: unclear_handler(x['request']['request'])),
}

delegation_branch = RunnableBranch(
    (lambda x: x['decision'].strip() == 'booker', branches["booker"]),  # 添加了 .strip()
    (lambda x: x['decision'].strip() == 'info', branches["info"]),  # 添加了 .strip()
    branches["unclear"]  # 'unclear' 或任何其他输出的默认分支
)

chain = {
                        "decision": coordinator_router_chain,
                        "request": RunnablePassthrough()
                    } | delegation_branch | (lambda x: x['output'])

print("--- 运行预订请求 ---")
request_a = "给我预订去伦敦的航班。"
result_a = chain.invoke({"request": request_a})
print(f"最终结果 A: {result_a}")

print("\n--- 运行信息请求 ---")
request_b = "意大利的首都是什么？"
result_b = chain.invoke({"request": request_b})
print(f"最终结果 B: {result_b}")

print("\n--- 运行不清楚的请求 ---")
request_c = "告诉我关于量子物理学的事。"
result_c = chain.invoke({"request": request_c})
print(f"最终结果 C: {result_c}")


--- 运行预订请求 ---

--- 委托给预订处理程序 ---
最终结果 A: 预订处理程序处理了请求：'给我预订去伦敦的航班。'。结果：模拟预订操作。

--- 运行信息请求 ---

--- 委托给信息处理程序 ---
最终结果 B: 信息处理程序处理了请求：'意大利的首都是什么？'。结果：模拟信息检索。

--- 运行不清楚的请求 ---

--- 委托给信息处理程序 ---
最终结果 C: 信息处理程序处理了请求：'告诉我关于量子物理学的事。'。结果：模拟信息检索。


In [2]:
from typing import Literal, TypedDict

from langchain_core.messages import SystemMessage, HumanMessage
from langgraph.constants import START
from langgraph.errors import GraphInterrupt
from langgraph.graph import StateGraph
from pydantic import BaseModel, Field

from utils.std_model import base_model

llm = base_model()


class Route(BaseModel):
    step: Literal["poem", "story", "joke"] = Field(
        None, description="The next step in the routing process"
    )


class State(TypedDict):
    input: str
    output: str
    decision: str


def llm_call_joke(state: State):
    """ create a joke """
    result = llm.invoke(state["input"])
    return {"output": result.content}


def llm_call_poem(state: State):
    """ create a poem """
    result = llm.invoke(state["input"])
    return {"output": result.content}


def llm_call_story(state: State):
    """ create a story """
    result = llm.invoke(state["input"])
    return {"output": result.content}


def llm_call_router(state: State):
    """ Route the input to the appropriate node """
    llm_with_tools = llm.with_structured_output(Route)
    route = llm_with_tools.invoke([
        SystemMessage(
            content="Route the input to story, joke, or poem based on the user's request."
        ),
        HumanMessage(
            content=state["input"],
        ),
    ])

    return {"decision": route.step}


def route_conditional(state: State):
    """ Route the input to the appropriate node """
    if state["decision"] == "poem":
        return llm_call_poem.__name__
    elif state["decision"] == "story":
        return llm_call_story.__name__
    elif state["decision"] == "joke":
        return llm_call_joke.__name__
    else:
        raise GraphInterrupt


graph_builder = StateGraph(State)
graph_builder.add_node(llm_call_router.__name__, llm_call_router)
graph_builder.add_node(llm_call_joke.__name__, llm_call_joke)
graph_builder.add_node(llm_call_poem.__name__, llm_call_poem)
graph_builder.add_node(llm_call_story.__name__, llm_call_story)

graph_builder.add_edge(START, llm_call_router.__name__)
graph_builder.add_conditional_edges(llm_call_router.__name__, route_conditional, [
    llm_call_poem.__name__,
    llm_call_story.__name__,
    llm_call_joke.__name__,
])

graph = graph_builder.compile()

graph.invoke({"input": "Write me a story about cats"})


{'input': 'Write me a story about cats',
 'output': "The world, to a cat named Jasper, was a symphony of scents and shadows. His kingdom was the neglected garden behind Mrs. Gable’s house, a wild tangle of mint and unkempt roses. He knew the precise warmth of the sun on the cracked flagstone path at 2:13 PM, and the exact rustle of a mouse in the compost heap at dusk. He was the sovereign of this small, fragrant domain.\n\nBut the throne was precarious. Winter was a lean hunter, and the summer rains flooded his best hiding spots beneath the shed. He had a coat of patchwork orange, a broken ear that folded like a tiny petal, and eyes the color of new honey. He was a survivor, a master of the quiet, calculated life.\n\nThen came the doorstep.\n\nIt appeared one Tuesday morning. A cardboard box, lined with an old, threadbare towel. And in it, two balls of fur, one the white of new snow, one the black of a starless night. Their eyes were still the milky blue of the newborn, and they mewed 